In [2]:
#imports
import pandas as pd
import numpy as np


In [47]:
### Men's Massey Ordinals

#data
massey_ordinals_m = pd.read_csv("data_2026/MMasseyOrdinals.csv")

#pre tournament rankings
pre_tournament_massey_m = massey_ordinals_m.query("RankingDayNum == 133")

#good subset of rankings
good_ratings_m = ["POM", "EBP", "MAS", "TRK", "HAS"]
g_pre_tournament_massey_m = pre_tournament_massey_m.query("Season >= 2016").loc[lambda df: df["SystemName"].isin(good_ratings_m)]

#group by and summarize
massey_m = g_pre_tournament_massey_m.groupby(["Season", "TeamID"]).agg(avg_massey_rank=("OrdinalRank", "mean")).reset_index()


In [ ]:
### Men's AP Rankings

#data
massey_ordinals_m = pd.read_csv("data_2026/MMasseyOrdinals.csv")

#preseason rankings
preseason_ap_m = massey_ordinals_m.query("SystemName == 'AP'") 
preseason_ap_m = preseason_ap_m[preseason_ap_m['RankingDayNum'] == preseason_ap_m.groupby('Season')['RankingDayNum'].transform('min')]
preseason_ap_m = preseason_ap_m.rename({'OrdinalRank': 'ap_preseason_rank'}, axis='columns')
preseason_ap_m = preseason_ap_m[['Season', 'TeamID', 'ap_preseason_rank']]

#pretournament ratings
pre_tournament_ap_m = massey_ordinals_m.query("SystemName == 'AP' & RankingDayNum == 133")
pre_tournament_ap_m = pre_tournament_ap_m.rename({'OrdinalRank': 'ap_pretournament_rank'}, axis='columns')
pre_tournament_ap_m = pre_tournament_ap_m[['Season', 'TeamID', 'ap_pretournament_rank']]

#combine
ap_m = pd.merge(preseason_ap_m, pre_tournament_ap_m, how="outer", on=["Season", "TeamID"])


,Season,TeamID,ap_preseason_rank,ap_pretournament_rank
0,2003,1104,2.0,NaN
1,2003,1112,1.0,2.0
2,2003,1158,25.0,NaN
3,2003,1163,9.0,23.0
4,2003,1166,23.0,15.0


In [29]:
### Get Stats Function

def get_stats(reg_stats):

    #Remove Loc, which causes problems
    reg_stats = reg_stats.drop(columns = 'WLoc')

    w_reg_stats = reg_stats.copy()
    l_reg_stats = reg_stats.copy()

    #For games team is winner
    w_reg_stats.columns = w_reg_stats.columns.str.replace(r'^L', 'opp_', regex=True)
    w_reg_stats.columns = w_reg_stats.columns.str.replace(r'^W', '', regex=True)

    #For games team is loser
    l_reg_stats.columns = l_reg_stats.columns.str.replace(r'^W', 'opp_', regex=True)
    l_reg_stats.columns = l_reg_stats.columns.str.replace(r'^L', '', regex=True)

    #combine the data
    reg_stats_pg = pd.concat([w_reg_stats, l_reg_stats])

    #per game percentages (need for tempo adjusted average percentages)
    reg_stats_pg = (
        reg_stats_pg.assign(margin = reg_stats_pg['Score'] - reg_stats_pg['opp_Score'])
            .assign(poss = lambda x: x['FGA'] - x['OR'] + x['TO'] + 0.475*x['FTA'])
            .assign(opp_poss = lambda x: x['opp_FGA'] - x['opp_OR'] + x['opp_TO'] + 0.475*x['opp_FTA'])
            .assign(eff=lambda x: x['Score'] / x['poss'])
            .assign(opp_eff=lambda x: x['opp_Score'] / x['opp_poss'])

            .assign(fg_per=lambda x: x['FGM'] / x['FGA'])
            .assign(fg_a_per=lambda x: x['FGA'] / x['poss'])
            .assign(thr_a_per=lambda x: x['FGA3'] / x['poss'])
            .assign(to_per=lambda x: x['TO'] / x['poss'])
            .assign(blk_per=lambda x: x['Blk'] / x['opp_FGA'])
            .assign(foul_rec_per=lambda x: x['opp_PF'] / x['poss'])
            .assign(foul_per=lambda x: x['PF'] / x['opp_poss'])
            .assign(or_per=lambda x: x['OR'] / (x['OR'] + x['opp_DR']))
            .assign(dr_per=lambda x: x['DR'] / (x['DR'] + x['opp_OR']))
        
            .assign(opp_fg_a_per=lambda x: x['opp_FGA'] / x['opp_poss'])
            .assign(opp_fg_per=lambda x: x['opp_FGM'] / x['opp_FGA'])
            .assign(opp_to_per=lambda x: x['opp_TO'] / x['opp_poss'])
    )

    f_reg_stats = reg_stats_pg.groupby(["Season", "TeamID"]).agg(
        avg_score=("Score", "mean"),
        avg_opp_score=("opp_Score", "mean"),
        avg_margin = ("margin", "mean"),
        avg_poss = ("poss", "mean"),
        avg_eff = ("eff", "mean"),
        avg_opp_eff = ("opp_eff", "mean"),
        avg_fg_per=("fg_per", "mean"),
        avg_m3=("FGM3", "mean"),
        avg_a3=("FGA3", "mean"),#
        avg_ftm=("FTM", "mean"),
        avg_fta=("FTA", "mean"),#
        avg_fg_a_per=("fg_a_per", "mean"),
        avg_thr_a_per=("thr_a_per", "mean"),
        avg_to_per=("to_per", "mean"),
        avg_blk_per=("blk_per", "mean"),
        avg_foul_rec_per=("foul_rec_per", "mean"),
        avg_foul_per=("foul_per", "mean"),
        avg_or_per=("or_per", "mean"),
        avg_dr_per=("dr_per", "mean"),
        avg_opp_fg_a_per=("opp_fg_a_per", "mean"),
        avg_opp_fg_per=("opp_fg_per", "mean"),
        avg_opp_to_per=("opp_to_per", "mean")
    ).reset_index()

    f_reg_stats = (
        f_reg_stats.assign(avg_thr_per = f_reg_stats['avg_m3'] / f_reg_stats['avg_a3'])
        .assign(ft_per=lambda x: x['avg_ftm'] / x['avg_fta'])
    )

    f_reg_stats = f_reg_stats.drop(columns = ['avg_a3', 'avg_m3', 'avg_ftm', 'avg_fta'])

    return f_reg_stats

In [52]:
### Get Stats

#data
reg_stats_m = pd.read_csv("data_2026/MRegularSeasonDetailedResults.csv")
reg_stats_w = pd.read_csv("data_2026/WRegularSeasonDetailedResults.csv")

#men's
f_reg_stats_m = get_stats(reg_stats_m)

#women's
f_reg_stats_w = get_stats(reg_stats_w)


f_reg_stats_m.sort_values('avg_eff', ascending = False).head()

,Season,TeamID,avg_score,avg_opp_score,avg_margin,avg_poss,avg_eff,avg_opp_eff,avg_fg_per,avg_fg_a_per,...,avg_blk_per,avg_foul_rec_per,avg_foul_per,avg_or_per,avg_dr_per,avg_opp_fg_a_per,avg_opp_fg_per,avg_opp_to_per,avg_thr_per,ft_per
8100,2026,1228,84.608696,67.173913,17.434783,67.848913,1.246914,1.000241,0.468165,0.896737,...,0.075889,0.284392,0.187353,0.396139,0.753633,0.944610,0.402631,0.107054,0.360839,0.789144
8214,2026,1345,83.636364,68.954545,14.681818,67.400000,1.239198,1.018392,0.511937,0.898195,...,0.049846,0.249377,0.210142,0.347530,0.753427,0.864360,0.431412,0.155145,0.388462,0.744565
5585,2019,1211,88.848485,65.060606,23.787879,71.668182,1.237302,0.908231,0.530947,0.846184,...,0.089764,0.260265,0.224976,0.301536,0.731357,0.860610,0.387227,0.188974,0.365057,0.767409
5455,2018,1437,87.058824,70.882353,16.176471,70.868382,1.229814,0.997585,0.505257,0.871187,...,0.065018,0.242710,0.222384,0.283763,0.736646,0.848740,0.435525,0.184785,0.397949,0.771285
7691,2025,1181,82.705882,61.911765,20.794118,67.393382,1.229000,0.913745,0.488629,0.877097,...,0.065625,0.244543,0.234826,0.335700,0.771748,0.853604,0.387337,0.158629,0.377193,0.784496


In [38]:
### Seed
seed_m = pd.read_csv("data_2026/MNCAATourneySeeds.csv")
seed_w = pd.read_csv("data_2026/WNCAATourneySeeds.csv")

seed_m = seed_m.assign(Seed = seed_m['Seed'].str[1:3])
seed_w = seed_w.assign(Seed = seed_w['Seed'].str[1:3])

In [55]:
### Combine

#Men's
combined_m = (
    seed_m
    .merge(ap_m, on=["Season", "TeamID"], how="outer")
    .merge(massey_m, on=["Season", "TeamID"], how="outer")
    .merge(f_reg_stats_m, on=["Season", "TeamID"], how="outer")
)

#Women's
combined_w = (
    seed_w
    .merge(f_reg_stats_w, on=["Season", "TeamID"], how="outer")
)